# 0 - Téléchargement des données

Ce notebook illustre les différentes fonctionnalités des clients du répertoire pour télécharger des données via l'API SDMX. Chaque client permet de télécharger des données depuis une source distincte (OCDE etc ...)

## Table des matières

0. [Importation des modules](#section-0)
1. [Téléchargement des données de l'OCDE](#section-1)
    - 1.1 [Enumération des dataflows](#section-1.1)
    - 1.2 [Description de la structure d'un dataflow](#section-1.2)
    - 1.3 [Requête basique avec positions](#section-1.3)
    - 1.4 [Requête avec noms de dimensions](#section-1.4)
    - 1.5 [Formats de réponse](#section-1.5)
    - 1.6 [Séparation des requêtes avec split dimensions](#section-1.6)
    - 1.7 [Utilisation de OECDQueryRequest](#section-1.7)
    - 1.8 [Filtre par date de mise à jour](#section-1.8)
    - 1.9 [Chargement depuis un fichier de configuration](#section-1.9)
    - 1.10 [Gestion des erreurs](#section-1.10)
    - 1.11 [Memento et conseils de performance](#section-1.11)

## 0 - Importation des modules <a id="section-0"></a>

Importation des modules nécessaires et configuration de l'environnement.

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import sys
import yaml
import pandas as pd
from datetime import datetime
from pathlib import Path

# Ajout du répertoire parent au path
sys.path.append('..')

# Modules du package
from macroforecast.datasets.sources import OECDClient, OECDQueryRequest, OECDResponseFormat

## 1 - Téléchargement des données de l'OCDE <a id="section-1"></a>

In [ ]:
# Initialisation du client
client = OECDClient()

### 1.1 - Enumération des dataflows <a id="section-1.1"></a>

La méthode `list_all_dataflows()` permet d'énumérer tous les dataflows disponibles pour toute les agencies d'une source de données.

In [ ]:
# Récupération de tous les dataflows
dataflows = client.list_all_dataflows()

# Affichage
print(f"Nombre total de dataflows: {len(dataflows)}")
print("\nPremiers dataflows:")
dataflows.head(10)

### 1.2 - Description de la structure d'un dataflow <a id="section-1.2"></a>

La méthode `get_structure()` pour d'extraire les dimensions d'un dataflow.

In [ ]:
# Extraction de la structure du dataflow KEI
structure = client.get_structure(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="4.0"
)

# Affichage
print(f"Agency: {structure.agency}")
print(f"Dataflow: {structure.dataflow}")
print(f"Nombre de dimensions: {structure.num_dimensions}")
print("\nDimensions:")
for dim in structure.dimensions:
    print(f"  Position {dim.position}: {dim.name} - {dim.description}")

### 1.3 - Requête basique avec positions <a id="section-1.3"></a>

La méthode `get_data()` permet de requêter les données d'un dataflow en appliquant des filtres sur les dimensions désignées par leur position (0, 1, 2...) dans l'URL de requête.

In [ ]:
# Requête avec positions numériques
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        0: ["FRA"],  # Position 0 = REF_AREA
        1: ["M"],    # Position 1 = FREQ
        2: ["LI"],   # Position 2 = MEASURE
    },
    start_period="2020"
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

### 1.4 - Requête avec noms de dimensions <a id="section-1.4"></a>

La méthode `get_data()` permet également de requêter les données d'un dataflow en appliquant des filtres sur les dimensions désignées par leur nom (plus lisible et robuste).

In [ ]:
# Requête avec noms de dimensions
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA", "DEU"],  # France et Allemagne
        "FREQ": "M",                  # Fréquence mensuelle
        "MEASURE": "LI",              # Leading Indicator
    },
    start_period="2020"
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Pays: {sorted(df['REF_AREA'].unique())}")
df.head()

### 1.5 - Formats de réponse <a id="section-1.5"></a>

L'API SDMX de l'OCDE supporte plusieurs formats de réponse. Le client permet de spécifier le format souhaité via le paramètre `format` de `OECDQueryRequest`.

In [ ]:
# Affichage des formats disponibles
print("Formats de réponse disponibles:")
for fmt in OECDResponseFormat:
    print(f"  - {fmt.name}: {fmt.value}")

In [ ]:
# Comparaison des formats CSV et CSV_LABELS
query_csv = OECDQueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA"],
        "FREQ": "M",
        "MEASURE": "LI"
    },
    start_period="2024",
    format=OECDResponseFormat.CSV  # Format avec codes uniquement
)

query_csv_labels = OECDQueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA"],
        "FREQ": "M",
        "MEASURE": "LI"
    },
    start_period="2024",
    format=OECDResponseFormat.CSV_LABELS  # Format avec labels lisibles (défaut)
)

# Exécution des requêtes
df_csv = client.execute_query(query_csv)
df_csv_labels = client.execute_query(query_csv_labels)

# Comparaison des colonnes
print("Colonnes avec CSV:")
print(f"  {list(df_csv.columns)}")
print("\nColonnes avec CSV_LABELS:")
print(f"  {list(df_csv_labels.columns)}")

# Aperçu des données
print("\n--- Format CSV (codes uniquement) ---")
print(df_csv[['REF_AREA', 'MEASURE', 'TIME_PERIOD', 'OBS_VALUE']].head(3).to_string(index=False))

print("\n--- Format CSV_LABELS (avec labels) ---")
# Affichage des colonnes de labels si présentes
label_cols = [c for c in df_csv_labels.columns if 'Reference area' in c or 'Measure' in c]
if label_cols:
    display_cols = ['REF_AREA'] + label_cols[:1] + ['MEASURE', 'TIME_PERIOD', 'OBS_VALUE']
    display_cols = [c for c in display_cols if c in df_csv_labels.columns]
    print(df_csv_labels[display_cols].head(3).to_string(index=False))

### 1.6 - Séparation des requêtes avec split dimensions <a id="section-1.6"></a>

Le paramètre `split_dimensions` de la méthode `get_data()` permet d'effectuer une requête distincte pour chaque valeur de la dimension spécifiée. Cela permet de gérer les requêtes volumineuses.

In [ ]:
# Requête avec split_dimensions
# Au lieu d'une seule requête pour 7 pays, on génère 7 requêtes séparées
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["CAN", "FRA", "DEU", "ITA", "JPN", "GBR", "USA"],
        "FREQ": "M",
        "MEASURE": "LI",
    },
    start_period="2020",
    split_dimensions=["REF_AREA"]  # Générera 7 requêtes séparées
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Pays: {sorted(df['REF_AREA'].unique())}")
print("\nStatistiques par pays:")
print(df.groupby('REF_AREA').size())

### 1.7 - Utilisation de OECDQueryRequest <a id="section-1.7"></a>

L'utilisation de l'objet `QueryRequest` permet la création et l'exécution d'une requête en manipulant ses paramètres de manière plus flexible qu'à travers l'utilisation des arguments de `get_data`.

In [ ]:
# Création d'une QueryRequest
query = OECDQueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA"],
        "FREQ": "M",
        "MEASURE": "LI"
    },
    start_period="2020"
)

# Affichage
print(f"Dataflow key: {query.get_dataflow_key()}")
print(f"Dimensions: {query.dimensions}")

# Exécution de la requête
df = client.execute_query(query)

# Affichage
print(f"\nNombre de lignes: {len(df)}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

### 1.8 - Filtre par date de mise à jour <a id="section-1.8"></a>

Le paramètre `updated_after` de `get_data()` / `OECDQueryRequest` permet la synchronisation incrémentale : la réponse ne contient alors que les observations insérées, mises à jour ou supprimées depuis l'instant indiqué. C'est utile pour ne re-télécharger que les données effectivement modifiées depuis le dernier téléchargement.

Il accepte une chaîne ISO-8601 (`"2020-01-01"`) ou un objet `datetime` (un `datetime` naïf est interprété en UTC). Ce mécanisme repose sur le paramètre `updatedAfter` du SDMX-CSV v2 et n'est donc disponible qu'avec le client en version v2 (le défaut).

In [ ]:
# Exemple 1 : date ancienne → renvoie les observations modifiées depuis 2020
query_old = OECDQueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    dimensions={"REF_AREA": ["FRA"], "FREQ": "M", "MEASURE": "LI"},
    start_period="2020",
    updated_after=datetime(2020, 1, 1)  # datetime naïf → interprété en UTC
)

df_old = client.execute_query(query_old)
print("--- updated_after=2020-01-01 ---")
print(f"Nombre d'observations modifiées depuis 2020: {len(df_old)}")

# Exemple 2 : maintenant → ne renvoie que les (rares) observations très récentes
query_now = OECDQueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    dimensions={"REF_AREA": ["FRA"], "FREQ": "M", "MEASURE": "LI"},
    start_period="2020",
    updated_after=datetime.now()
)

df_now = client.execute_query(query_now)
print("\n--- updated_after=datetime.now() ---")
print(f"Nombre d'observations modifiées depuis maintenant: {len(df_now)}")

### 1.9 - Chargement des requêtes depuis un fichier de configuration <a id="section-1.9"></a>

Les requêtes à effectuer peuvent également êre instanciées depuis un fichier YAML de configuration de la manière suivante.

In [ ]:
# Chargement de la configuration
config_path = Path('../config/datasets/oecd.yaml')
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("Requêtes configurées:")
if 'queries' in config:
    # Enumération des requêtes disponibles
    for query_name in config['queries'].keys():
        print(f"  - {query_name}")

    # Exemple: chargement et exécution de la première requête
    query_cfg = config['queries'][list(config['queries'].keys())[0]]

    # Affichage des caractéristiques de la requête
    print(f"\nConfiguration de 'kei_g7_monthly':")
    print(f"  Agency: {query_cfg['agency']}")
    print(f"  Dataflow: {query_cfg['dataflow']}")
    print(f"  Dimensions: {list(query_cfg['dimensions'].keys())}")

    # Création de la QueryRequest depuis la config
    query = OECDQueryRequest(
        agency=query_cfg['agency'],
        dataflow=query_cfg['dataflow'],
        version=query_cfg.get('version', '+'),
        dimensions=query_cfg['dimensions'],
        start_period=query_cfg.get('start_period'),
        end_period=query_cfg.get('end_period'),
        split_dimensions=query_cfg.get('split_dimensions')
    )

    # Affichage
    print(f"\nQueryRequest créée: {query.get_dataflow_key()}")
    print("\nPour exécuter la requête, décommentez la ligne suivante:")
    print("# df = client.execute_query(query)")

else:
    print("⚠ Section 'queries' non trouvée dans la configuration")

### 1.10 - Gestion des erreurs <a id="section-1.10"></a>

Démonstration de la gestion des erreurs avec des requêtes invalides.

In [ ]:
# Test 1: Agency invalide
print("Test 1: Agency invalide")
try:
    df = client.get_data(
        agency="INVALID_AGENCY",
        dataflow="INVALID_DATAFLOW",
        dimensions={"REF_AREA": ["FRA"]}
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:100]}")

# Test 2: Dimension invalide
print("\nTest 2: Dimension invalide")
try:
    df = client.get_data(
        agency="OECD.SDD.STES",
        dataflow="DSD_KEI@DF_KEI",
        dimensions={"INVALID_DIM": ["VALUE"]}
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:100]}")

print("\n✓ Tests de gestion d'erreurs terminés")

### 1.11: Memento et conseils de performance <a id="section-1.11"></a>

Meilleures pratiques pour optimiser les téléchargements de données OECD.

### 1. Rate Limiting
- Le client OECD implémente automatiquement un [rate limiting](https://www.oecd.org/fr/data/insights/data-explainers/2024/11/Api-best-practices-and-recommendations.html) de **60 requêtes par heure**
- Aucune action requise, c'est géré automatiquement

### 2. Split Dimensions
- Utiliser `split_dimensions` pour les requêtes avec beaucoup de valeurs
- Génère plusieurs petites requêtes au lieu d'une grosse requête
- Exemple: 7 pays × 14 transactions = 98 requêtes avec `split_dimensions=["REF_AREA", "TRANSACTION"]`

```python
# Séparation en plusieurs requêtes
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    dimensions={"REF_AREA": ["CAN", "FRA", "DEU", "ITA", "JPN", "GBR", "USA"]},
    split_dimensions=["REF_AREA"]
)
```

### 3. Filtrage par date de mise à jour
- Utiliser `updated_after` pour ne récupérer que les observations modifiées depuis une date (SDMX-CSV v2)
- Accepte une chaîne ISO-8601 ou un objet `datetime`
- Idéal pour la synchronisation incrémentale (ne re-télécharger que ce qui a changé)

```python
from datetime import datetime

# Ne récupère que les observations modifiées depuis le 15 décembre 2024
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    dimensions={"REF_AREA": ["FRA"]},
    updated_after=datetime(2024, 12, 15)
)
```

### 4. Formats de réponse
- **CSV_LABELS** (défaut): CSV avec labels lisibles - recommandé
- **CSV**: CSV avec codes - plus léger mais moins lisible
- **JSON**: Format JSON - plus flexible mais plus lourd

```python
from macroforecast.datasets.sdmx import ResponseFormat

query = QueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    format=ResponseFormat.CSV  # Plus rapide si vous connaissez les codes
)
```

### 5. Gestion des doublons
- Par défaut: `on_duplicate="warn"` (affiche un warning)
- Options: `"ignore"`, `"warn"`, `"raise"`

```python
query = QueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    on_duplicate="raise"  # Lève une exception en cas de doublon
)
```

### 6. Limitation temporelle
- Utiliser `start_period` et `end_period` pour limiter la période
- Ou `last_n_observations` pour ne récupérer que les N dernières observations

```python
# Seulement les 12 derniers mois
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    dimensions={"REF_AREA": ["FRA"]},
    last_n_observations=12
)
```

## Conclusion

Ce notebook a démontré toutes les fonctionnalités principales :

- du client OECD (énumération des dataflows, extraction de leur structure, exécution de requête avec différents paramètres ...) ; 

Pour plus d'informations, consulter:
- Le fichier `macroforecast/datasets/oecd.py`
- La configuration `config/datasets/oecd.yaml`